In [ ]:
from datasets import load_dataset
import os
# Get current working directory
base_path = os.getcwd()

data_path = os.path.join(base_path, '..', 'data', 'WAVA') 
data = load_dataset("audiofolder", data_dir = data_path)

data

### 1) Extract the audio filename 

In [ ]:
import os
from tqdm import tqdm  # tqdm is used for the progress bar

# Assuming 'data' is your dataset loaded with Hugging Face datasets
audio_filenames = []

# Loop through the dataset with a progress bar
for item in tqdm(data['train'], desc="Extracting file names"):
    audio_path = item['audio']['path']
    audio_filename = os.path.basename(audio_path)     # Extract the file name
    audio_name = os.path.splitext(audio_filename)[0]  # Remove the .wav extension
    audio_filenames.append(audio_name)
    
audio_filenames

### 2) Check corresponding Transcript file and Wav file

In [ ]:
# The main purporse of this function is to extract transcripts from a given file and map them to audio filenames.
#  INPUT  -> 000100001	一八七二年一月十日 ，
#  OUTPUT 
#  A file -> 000100001 ,  B file -> 一八七二年一月十日

def extract_transcripts(audio_filenames, transcript_file, filename_output, text_output):
    """
    Extract transcript lines where filename exists in audio_filenames.
    Write matched filenames and their cleaned transcripts to separate files.
    """
    audio_to_text = {}

    try:
        with open(transcript_file, 'r', encoding='utf-8') as file:
            lines = file.readlines()
    except FileNotFoundError:
        print(f"[ERROR] Transcript file not found: {transcript_file}")
        return {}

    for line in lines:
        parts = line.strip().split('\t', 1)
        if len(parts) != 2:
            continue  # Skip malformed lines
        filename, text = parts
        text_no_spaces = text.replace(' ', '')
        if filename in audio_filenames:
            audio_to_text[filename] = text_no_spaces

    # Save filenames (with .WAV suffix) to one file
    with open(filename_output, 'w', encoding='utf-8') as f_out:
        for filename in audio_to_text:
            f_out.write(f"{filename}.WAV\n")

    # Save cleaned text to another file
    with open(text_output, 'w', encoding='utf-8') as t_out:
        for text in audio_to_text.values():
            t_out.write(f"{text}\n")

    print(f"[INFO] Extracted {len(audio_to_text)} entries from {os.path.basename(transcript_file)}")
    return audio_to_text


# Get correct script paths
actual_script = os.path.join(base_path, '..', 'data', 'SCRIPT', 'Actual_text', 'total_transcript.txt')
suppose_script = os.path.join(base_path, '..', 'data', 'SCRIPT', 'Suppose_text', 'total_transcript.txt')

# Output paths
output_pairs = [
    (actual_script, 'actual_wav.txt', 'actual_transcript.txt'),
    (suppose_script, 'suppose_wav.txt', 'suppose_transcript.txt')
]

# Run processing
for transcript_file, filename_out, text_out in output_pairs:
    extract_transcripts(audio_filenames, transcript_file, filename_out, text_out)

print("[DONE] All transcripts processed.")


#### 2.2) Filtering the dataset with correct audio and transcriptions

In [ ]:
import os
from tqdm import tqdm

# Transcript_file is the file with this format:
# 000100001.WAV
# This is used to check if the audio file is corresponding to the pinyin Notations

def filter_data_based_on_transcripts(data, transcript_file):
    # Read all lines from the transcript file and strip whitespace
    with open(transcript_file, 'r', encoding='utf-8') as file:
        lines = [line.strip() for line in file.readlines()]
    
    # Initialize a list to store indices of items to keep
    indices_to_keep = []
    
    # Loop through each item in the dataset
    for idx, item in tqdm(enumerate(data['train']), desc="Filtering dataset"):
        audio_path = item['audio']['path']
        audio_filename = os.path.basename(audio_path)  # Extract the file name
        
        # If the filename is in the transcript file, keep the index
        if audio_filename in lines:
            indices_to_keep.append(idx)
    
    # Filter the dataset to keep only the indices we want
    filtered_data = data['train'].select(indices_to_keep)
    
    return filtered_data

# Example usage
transcript_file1 = 'actual_wav.txt'  
transcript_file2 = 'suppose_wav.txt'
data['train'] = filter_data_based_on_transcripts(data, transcript_file1)
data['train'] = filter_data_based_on_transcripts(data, transcript_file2)


data
    

### Transcript SENTENCE conversion and insertion 

In [ ]:
# Step 1: Read the cleaned sentences from User choices         # Important 
sentence_actual = 'actual_transcript.txt'                    # Important !!!!!!
sentence_suppose = 'suppose_transcript.txt'             # Important  !!!!!!!!

actual = []
suppose = []
with open(sentence_actual, 'r', encoding='utf-8') as f:
    actual = f.readlines()

with open(sentence_suppose, 'r', encoding='utf-8') as f:
    suppose = f.readlines()
    
# Step 2: Clean up the sentence list (strip newline characters)
actual = [sentence.strip() for sentence in actual]
suppose = [sentence.strip() for sentence in suppose]

# Step 3: Add the 'sentence' feature to the existing dataset
# We can use the `map` function to add the sentences to each row in the dataset
def add_sentences(example, idx):
    example['sentence_speaker_said'] = actual[idx]
    example['sentence_supposed_said'] = suppose[idx]
    return example

# Apply the 'add_sentences' function to each row of the dataset
updated_dataset = data['train'].map(add_sentences, with_indices=True)

# Update the dataset
data['train'] = updated_dataset

print(data['train'][13])
data

### TONE insertion with tone 5 change to tone 0

In [ ]:
# Step 1: Read the cleaned sentences from User choices          Important 
tone_actual_file = 'tone_actual.txt'                 # Important !!!!!!
tone_suppose_file = 'tone_suppose.txt'          # Important  !!!!!!!!

tone_actual = []
tone_suppose = []
with open(tone_actual_file, 'r', encoding='utf-8') as f:
    tone_actual = f.readlines()

with open(tone_suppose_file, 'r', encoding='utf-8') as f:
    tone_suppose = f.readlines()
    
# Step 2: Clean up the sentence list (strip newline characters)
tone_actual = [sentence.strip() for sentence in tone_actual]
tone_suppose = [sentence.strip() for sentence in tone_suppose]

# Step 3: Add the 'sentence' feature to the existing dataset
# We can use the `map` function to add the sentences to each row in the dataset
def add_sentences(example, idx):
    example['tone_pinyin_actual'] = tone_actual[idx]
    example['tone_pinyin_suppose'] = tone_suppose[idx]
    return example

# Apply the 'add_sentences' function to each row of the dataset
updated_dataset = data['train'].map(add_sentences, with_indices=True)

# Update the dataset
data['train'] = updated_dataset

print(data['train'][28]['tone_pinyin_actual'])
print(data['train'][28]['tone_pinyin_suppose'])

# Define a helper function to replace '5' with '0' in specified fields
def replace_five_with_zero(batch):
    batch['tone_pinyin_actual'] = batch['tone_pinyin_actual'].replace('5', '0')
    batch['tone_pinyin_suppose'] = batch['tone_pinyin_suppose'].replace('5', '0')
    return batch

# Apply the function to each subset of the dataset
for subset in ['train']:
    data[subset] = data[subset].map(replace_five_with_zero)
    
data

### Add Pinyin transcript into the dataset

In [ ]:
import re
from dragonmapper import hanzi

def remove_tones(ipa_string):
    # Define a pattern to match only tone markers (˥, ˧, ˩, etc.)
    pattern = r'[˥˧˩˦˨×]+'
    
    # Remove the tone markers using re.sub()
    ipa_string_no_tones = re.sub(pattern, '', ipa_string)
    
    return ipa_string_no_tones

def remove_punctuation(input_string):
    # Updated regex pattern to include a wide range of punctuation marks and spaces
    pattern = r"[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~·•《》「」『』【】…（）、；：！？——‘’“”，‧。“”·：、/ㄟＰ|・／－〉〈─□ΛSPK]+"
    
    # Remove the unwanted characters
    cleaned_string = re.sub(pattern, '', input_string)
    
    return cleaned_string

# List of exceptions to keep together
exceptions = ['pʰ', 'ts', 'tsʰ', 'tʰ', 'ʈʂ', 'ʈʂʰ', 'tɕ', 'tɕʰ', 'kʰ', 'ɑɻ', 'ai', 'ei', 'ɑʊ', 'oʊ', 'ia', 'iɛ', 'wa', 'wɔ', 'ɥœ', 'iɑʊ', 'ioʊ', 'wai', 'weɪ']
def convert_phoneme(input_string):
    try:
        # Remove punctuation
        input_string = remove_punctuation(input_string)
        
        # Convert to IPA
        ipa_result = hanzi.to_ipa(input_string, delimiter=' ', all_readings=False, container='[]')
        ipa_result = ipa_result.replace('j', 'i').replace('ɪ', 'i').replace('ń', 'ən')  # .replace('ń', 'ən') Only For ‘嗯’ Here

        # Remove tones and spaces from the IPA result
        ipa_result = remove_tones(ipa_result)
        ipa_result = ' '.join(ipa_result.split())  # Removing all spaces
        
        symbols = list(ipa_result)
        result = []
        i = 0
        
        while i < len(symbols):
            
            # Check for syllable 'ɻ' and replace it with 'ɑɻ'
            if symbols[i] == 'ɻ':
                result.append('ɑɻ')
                i += 1  # Move to the next character
                
            # Try matching 3-character exceptions first
            elif i < len(symbols) - 2 and (symbols[i] + symbols[i + 1] + symbols[i + 2]) in exceptions:
                result.append(symbols[i] + symbols[i + 1] + symbols[i + 2])
                i += 3  # Skip the next two characters since we've processed them as part of the exception
            # Try matching 2-character exceptions
            elif i < len(symbols) - 1 and (symbols[i] + symbols[i + 1]) in exceptions:
                result.append(symbols[i] + symbols[i + 1])
                i += 2  # Skip the next character since we've processed it as part of the exception
            else:
                # No match, so treat the current character as a separate symbol
                result.append(symbols[i])
                i += 1
        
        return ' '.join(result)

    except ValueError as e:
        print(f"Error processing syllable: {input_string}")
        print(f"Error details: {e}")
        raise


# Define a function to add the 'transcript' for actual and suppose
def add_transcript_column(example):
    example['transcript_IPA_actual'] = convert_phoneme(example['sentence_speaker_said'])
    example['transcript_IPA_suppose'] = convert_phoneme(example['sentence_supposed_said'])
    return example


# Apply this function to add 'transcript' to the 'train', 'validation', and 'test' datasets
data['train'] = data['train'].map(add_transcript_column)


#### Split the dataset

In [ ]:
from datasets import DatasetDict

size_split = 0.3
# Step 1: Split the first 70% for training
train_test_split = data['train'].train_test_split(test_size=size_split, shuffle=False)

# Step 2: Split the remaining 30% into test and validation (14% and 15%)
test_valid_split = train_test_split['test'].train_test_split(test_size=0.5, shuffle=False)

# Step 3: Combine into a new DatasetDict with train, test, and validation sets
split_dataset = DatasetDict({
    'train': train_test_split['train'],         # 70% of the data
    'test': test_valid_split['train'],          # 15% of the data
    'validation': test_valid_split['test']      # 15% of the data
})
data = split_dataset 
data

In [ ]:
new_path = 'PATH/OF/YOUR '# Define your new path here
data.save_to_disk(new_path)